# XGB

In [ ]:
import sklearn as skl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prep3 import DataPreparation
from sklearn.metrics import f1_score

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedShuffleSplit
from sklearn import tree

from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.metrics import classification_report
%matplotlib inline

In [ ]:
prep = DataPreparation()

train_db = pd.read_csv("dataset/train_radiomics_hipocamp.csv")
test_db = pd.read_csv("dataset/test_radiomics_hipocamp.csv")
control_db = pd.read_csv("dataset/train_radiomics_occipital_CONTROL.csv")

train_db = prep.prep_train(train_db)
test_db = prep.transform(test_db)
control_db = prep.transform(control_db)

In [ ]:
train_db.info()

In [ ]:
test_db.info()

In [ ]:
X = train_db.drop(columns=["Transition"]) 
y = train_db["Transition"]  

In [ ]:
X_train = pd.read_csv('prep3/train_X.csv')
X_test = pd.read_csv('prep3/test_X.csv')
y_train = pd.read_csv('prep3/train_y.csv')
y_test = pd.read_csv('prep3/test_y.csv')

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y) 
print("R?tulos ap?s codifica??o:", label_encoder.classes_)

clf = XGBClassifier(
    random_state=2024, 
    eval_metric='logloss'
)

param_grid = {
    'n_estimators': [50],
    'max_depth': [3],               
    'learning_rate': [0.3],    
    'subsample': [0.8],              
    'colsample_bytree': [0.6],       
    'min_child_weight': [1],        
    'gamma': [0.1]                
}


cv_strategy = StratifiedKFold(n_splits=10)

grid_search = GridSearchCV(
    estimator=clf, 
    param_grid=param_grid, 
    scoring='f1_macro',  
    cv=cv_strategy,        
    verbose=1,              
    refit=True,
    n_jobs=-1
)

grid_search.fit(X, y_encoded)

print("Melhores par?metros encontrados:", grid_search.best_params_)

best_model = grid_search.best_estimator_

print("F1 Score m?dio com valida??o cruzada:", grid_search.best_score_)

y_train_encoded = label_encoder.transform(y_train)  
best_model.fit(X_train, y_train_encoded)

y_test_encoded = label_encoder.transform(y_test)  
y_pred_encoded = best_model.predict(X_test)

# Decodificar as previs?es para r?tulos originais
y_pred = label_encoder.inverse_transform(y_pred_encoded)

# Matriz de confus?o
print("Matriz de Confus?o:")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
plt.show()

# Relat?rio de classifica??o
print("\nRelat?rio de Classifica??o:")
print(classification_report(y_test, y_pred))


### Testar Controlo

In [ ]:
CX = control_db.drop(columns=["Transition"]) 
Cy = control_db["Transition"]  

Cy_encoded = label_encoder.transform(Cy)

control_pred = best_model.predict(CX)

control_f1_score = f1_score(Cy_encoded, control_pred, average='macro')
print("F1-Score no conjunto de controle:", control_f1_score)

# Matriz de Confus?o
ConfusionMatrixDisplay.from_predictions(Cy_encoded, control_pred, cmap='Blues')
plt.show()


### Guardar o melhor modelo do XGB

In [ ]:
y_encoded_full = label_encoder.fit_transform(y)

best_model.fit(X, y_encoded_full)

test_predictions_encoded = best_model.predict(test_db)

test_predictions = label_encoder.inverse_transform(test_predictions_encoded)

row_ids = range(1, len(test_predictions) + 1)
output_df = pd.DataFrame({
    "RowId": row_ids,
    "Result": test_predictions
})

output_df.to_csv("resultados/xgb3.csv", index=False)